# KNOTS-03 — Le lake `knot_lean` par ses déclarations (compagnon formel)

Compagnon du **Lean-17** (« Conway, les Nœuds et la Preuve de Piccirillo »). Là où Lean-17 raconte l'histoire mathématique (Fox 3-colorabilité, mutants, Piccirillo 2020, Lidman 2026) et interroge `Conway.lean` et `Lidman.lean`, ce notebook interroge les modules que Lean-17 ne cite pas — **`Basic.lean`** (les structures fondamentales), **`Invariant.lean`** (la chaîne 3-colorabilité), **`Reidemeister.lean`** (les moves R1/R2/R3 et leurs murs) et **`MathlibPrerequisites.lean`** (la feuille de route des prérequis Mathlib) — et mesure l'état de la formalisation : déclarations par module, sorries réels vs prose, axiomes admis.

Kernel Python par choix : le kernel natif `lean4-wsl` est gelé en attendant la mise à jour du binaire `repl` (#11874 — construit pour v4.30.0, lakes en v4.33.0). La lecture des sources reste réelle : chaque cellule ouvre le fichier `.lean` du lake et en extrait les déclarations, jamais une copie.

Voir `knot_lean/README.md` pour l'état détaillé du corridor Reidemeister.

## 1. Le lac : deux langues

Le lake suit la convention i18n #4980 : chaque module français a son sibling `_en` byte-identique sur le code, docstrings traduites. On inventorie les modules FR et leurs miroirs.

In [1]:
import re
from pathlib import Path

LAKE = Path('knot_lean')
modules = sorted(p for p in (LAKE / 'Knots').glob('*.lean') if not p.stem.endswith('_en'))
print(f"Modules FR du lake : {len(modules)}")
for p in modules:
    en = p.with_name(p.stem + '_en.lean')
    sib = 'oui' if en.exists() else 'ABSENT'
    print(f"  {p.name:28} {len(p.read_text(encoding='utf-8').splitlines()):>5} lignes   sibling _en : {sib}")

Modules FR du lake : 23
  AlexanderTrivial.lean         3833 lignes   sibling _en : oui
  ArcPartition.lean              951 lignes   sibling _en : oui
  Basic.lean                     680 lignes   sibling _en : oui
  BoundedEnumeration.lean        197 lignes   sibling _en : oui
  Conway.lean                     26 lignes   sibling _en : oui
  ConwayKTMutation.lean          236 lignes   sibling _en : oui
  ConwayNotation.lean            558 lignes   sibling _en : oui
  ConwayPD.lean                  160 lignes   sibling _en : oui
  FigureEight.lean               224 lignes   sibling _en : oui
  Invariant.lean                3709 lignes   sibling _en : oui
  Jones.lean                     651 lignes   sibling _en : oui
  Khovanov.lean                  500 lignes   sibling _en : oui
  Lidman.lean                   1254 lignes   sibling _en : oui
  MathlibPrerequisites.lean      138 lignes   sibling _en : oui
  Mutation.lean                  416 lignes   sibling _en : oui
  Redraw.lean   

  ReidemeisterAny.lean          3356 lignes   sibling _en : oui
  ReidemeisterCombinatorial.lean   991 lignes   sibling _en : oui
  ReidemeisterInvariance.lean    672 lignes   sibling _en : oui
  ReidemeisterMoves.lean         910 lignes   sibling _en : oui
  Slice.lean                     153 lignes   sibling _en : oui
  Tangle.lean                    485 lignes   sibling _en : oui


### Lecture du résultat
Chaque module FR de la liste a son miroir `_en` — la convention i18n est respectée sur ce lake (miroirs §9), et la cellule le vérifie module par module. `Invariant.lean` est le plus gros en lignes (la chaîne 3-colorabilité, §3), `MathlibPrerequisites` le plus concis — la feuille de route pure de §7.

Le lake a changé depuis la première rédaction, et la liste le montre : il compte désormais dix-neuf modules FR. `Conway.lean` n'est plus le monolithe d'origine mais un *aggregator* de vingt-six lignes (plan Option A, #18397) — sa matière a essaimé dans `Mutation`, `ConwayPD`, `ArcPartition` et `AlexanderTrivial` — et `ConwayNotation`, `ConwayKTMutation` et `Tangle` sont venus s'ajouter.

La sortie liste les modules FR, tous marqués `sibling _en : oui`, avec `Invariant.lean` en tête du classement par taille et `Conway.lean` réduit à son squelette.


## 2. `Basic.lean` — les structures fondamentales

Tout le lake repose sur quatre structures : `Crossing`, `PDCrossing`, `KnotDiagram`, `Knot` (+ `Link`). Les nœuds canoniques (`unknot`, `trefoil`, `figureEight`) y sont définis, ainsi que le miroir et le nombre de croisements.

In [2]:
# Declarations publiques de Basic.lean, extraites du fichier reel
DECL_RE = re.compile(r'^(structure|theorem|lemma|def|instance|abbrev)\s+([A-Za-z_][A-Za-z0-9_.\'\?]*)')

def decls_of(path):
    out = []
    for i, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        m = DECL_RE.match(line)
        if m:
            out.append((i, m.group(1), m.group(2)))
    return out

basic = decls_of(LAKE / 'Knots' / 'Basic.lean')
print(f"Basic.lean : {len(basic)} declarations")
for ln, kind, name in basic:
    print(f"  L{ln:>4}  {kind:9} {name}")

Basic.lean : 34 declarations
  L  53  structure Crossing
  L  81  structure PDCrossing
  L  90  structure KnotDiagram
  L 109  structure Knot
  L 119  structure Link
  L 127  def       Knot.toLink
  L 137  def       unknotDiagram
  L 141  def       unknot
  L 165  def       trefoilDiagram
  L 173  def       trefoil
  L 191  def       figureEightDiagram
  L 200  def       figureEight
  L 208  def       mirrorCrossing
  L 214  def       Knot.mirror
  L 226  def       Knot.crossingNumberOfDiagram
  L 252  def       Knot.crossingNumber
  L 261  def       KnotDiagram.edges
  L 265  def       KnotDiagram.numCrossings
  L 302  def       KnotDiagram.wf
  L 311  theorem   unknot_wf
  L 315  theorem   trefoil_wf
  L 319  theorem   figureEight_wf
  L 343  theorem   mirror_unknot_wf
  L 350  theorem   mirror_trefoil_wf
  L 356  theorem   mirror_figureEight_wf
  L 392  theorem   mirrorCrossing_perm
  L 413  theorem   mirrorCrossing_preserves_count
  L 424  theorem   count_lift_append
  L 449  theor

### Lecture du résultat
Les trois nœuds canoniques du lake sont des `def` explicites (`unknotDiagram`, `trefoilDiagram`, `figureEightDiagram` puis leurs objets `Knot`). `wf` (well-formed) est un `Bool` : la correction du PD-code est décidable et testée — `unknot_wf` en est la preuve minimale.

La sortie compte **34 déclarations** : cinq `structure` en tête (L53–L119 : `Crossing`, `PDCrossing`, `KnotDiagram`, `Knot`, `Link`), puis les `def` des nœuds canoniques et quatorze `theorem` de miroir et de bien-fondé.

## 3. `Invariant.lean` — la chaîne 3-colorabilité

C'est le cœur pédagogique du lake : la définition `IsTricolorable` et le théorème-cible `tricolorable_invariant` (« la 3-colorabilité de Fox est invariante par moves de Reidemeister »), avec sa décomposition en bras ascendants R1/R2 et ses murs nommés.

In [3]:
# Les declarations cibles de la chaine tricolorabilite
inv = decls_of(LAKE / 'Knots' / 'Invariant.lean')
print(f"Invariant.lean : {len(inv)} declarations\n")
CIBLES = ['IsTricolorable', 'triColorFoxCondition_iff_sum_mod_three',
          'tricolorable_invariant', 'tricolorable_forward_r1',
          'tricolorable_forward_r2_up', 'r2_append_only_wall', 'r3_determined_wall']
for ln, kind, name in inv:
    if any(name.startswith(c) or c in name for c in CIBLES):
        print(f"  L{ln:>4}  {kind:9} {name}")

Invariant.lean : 86 declarations

  L 234  def       IsTricolorable
  L 304  instance  IsTricolorable.decidable
  L 338  theorem   triColorFoxCondition_iff_sum_mod_three
  L 715  theorem   tricolorable_forward_r1
  L 960  theorem   tricolorable_forward_r2_up
  L1088  theorem   r2_append_only_wall
  L1241  theorem   r3_determined_wall
  L1380  theorem   tricolorable_invariant_fails_under_pr1_model
  L3356  theorem   tricolorable_invariant_r2_connected
  L3642  theorem   tricolorable_invariant_r3_connected
  L3657  theorem   tricolorable_invariant


### Lecture du résultat
La chaîne expose la discipline de preuve du lake : `tricolorable_invariant` (le théorème-cible) est décomposé en bras `tricolorable_forward_r1` et `tricolorable_forward_r2_up`, chacun borné par des **murs nommés** (`r2_append_only_wall`, `r3_determined_wall`) — des `theorem` qui énoncent l'obligation restante au lieu de la cacher dans un `sorry` anonyme. La bi-implication R1 est complète (forward + backward, #11227).

La sortie totalise **86 déclarations** et en isole onze lignes cibles, de `IsTricolorable` (ligne 234) au théorème-cible `tricolorable_invariant` (ligne 3657), en passant par le mur du modèle PR1 `tricolorable_invariant_fails_under_pr1_model` (ligne 1380).

## 4. Sorries réels vs prose — l'instrument juste

Le README du lake documente deux comptes : `grep -c sorry` naïf (compte la prose) et le mode `real` du CI (strippe commentaires, compte mot-bounded). Un `grep` naïf sur `Invariant.lean` rend un total beaucoup plus élevé que le réel. On mesure les deux.

In [4]:
# Les deux comptes, mesures sur les fichiers reels
def count_sorry(path, mode):
    text = path.read_text(encoding='utf-8')
    if mode == 'naive':
        return text.count('sorry')
    # mode real : strippe -- et /- -/, compte le mot bounded
    stripped = re.sub(r'/-([^-]|-[^/])*-/', ' ', text, flags=re.S)
    stripped = re.sub(r'--.*', ' ', stripped)
    return len(re.findall(r'\bsorry\b', stripped))

rows = []
for p in modules:
    rows.append([p.stem, count_sorry(p, 'naive'), count_sorry(p, 'real')])
rows.append(['TOTAL', sum(r[1] for r in rows), sum(r[2] for r in rows)])
import pandas as pd
df = pd.DataFrame(rows, columns=['module', 'grep naif', 'sorry reels'])
df

,module,grep naif,sorry reels
0,AlexanderTrivial,2,0
1,ArcPartition,0,0
2,Basic,3,0
3,BoundedEnumeration,1,0
4,Conway,0,0
5,ConwayKTMutation,0,0
6,ConwayNotation,0,0
7,ConwayPD,0,0
8,FigureEight,0,0
9,Invariant,6,0


### Lecture du résultat
L'écart naive/réel illustre le piège documenté dans les règles du dépôt : les docstrings FR documentent précisément l'absence de preuve (« mur R2 », « restent 2 sorries ») — un `grep` naïf compte cette documentation. Le CI gate sur le compte réel. Un instrument qui surestime la dette fausse l'arbitrage de ce qu'il faut prouver ensuite.

Chiffré tel quel : le total affiché est **38 `sorry` en grep naïf contre 8 réels**, et la dette réelle ne touche que trois modules — `Slice` (4), `Lidman` (2) et `Reidemeister` (2) ; `Invariant.lean` affiche 6 en naïf pour 0 réel.

## 5. `Reidemeister.lean` — les moves et le corridor sémantique

Les moves R1/R2/R3 y sont des `Prop` explicites (`Reidemeister1`, `Reidemeister2`, ...) avec symétries prouvées. Le corridor #8696 (5 PRs mergées) y a remplacé le move-surgery `with` par des égalités de champs. Des sorries réels subsistent : `reidemeister_theorem` (topologie PL des 3-variétés, hors Mathlib actuel — documenté INTRINSIC).

In [5]:
# Les moves Reidemeister et leurs proprietes
reid = decls_of(LAKE / 'Knots' / 'Reidemeister.lean')
print(f"Reidemeister.lean : {len(reid)} declarations\n")
for ln, kind, name in reid:
    tag = ''
    if name.startswith('Reidemeister'):
        tag = '  <-- move'
    print(f"  L{ln:>4}  {kind:9} {name}{tag}")

Reidemeister.lean : 36 declarations

  L  87  def       Reidemeister1  <-- move
  L  97  theorem   Reidemeister1.symm  <-- move
  L 145  def       Reidemeister1'  <-- move
  L 164  theorem   Reidemeister1'.implies_reidemeister1  <-- move
  L 217  def       PDCrossing.isRenameOf
  L 229  def       PDCrossing.hasEdge
  L 262  def       Reidemeister1Connected  <-- move
  L 286  theorem   reidemeister1Connected_satisfiable
  L 320  theorem   Reidemeister1Connected.numEdges_succ  <-- move
  L 328  theorem   Reidemeister1Connected.numCrossings_succ  <-- move
  L 337  theorem   Reidemeister1Connected.shares_edge  <-- move
  L 350  theorem   Reidemeister1Connected.crossings_eq  <-- move
  L 374  def       Reidemeister2  <-- move
  L 383  theorem   Reidemeister2.symm  <-- move
  L 414  def       PDCrossing.isDoubleRenameOf
  L 444  def       Reidemeister2Connected  <-- move
  L 466  theorem   reidemeister2Connected_satisfiable
  L 494  theorem   Reidemeister2Connected.numEdges_succ  <-- move
  

### Lecture du résultat
Chaque move a sa `.symm` prouvée (la symétrie n'est pas une hypothèse cachée). `Reidemeister1Connected` vient avec ses théorèmes de comptage (`numEdges_succ`, `numCrossings_succ`) : le modèle R1-connecté rend le move *localement vérifiable* — c'est ce qui a permis le corridor.

La sortie compte **36 déclarations** : les trois moves de base `Reidemeister1`/`Reidemeister2`/`Reidemeister3` (lignes 87, 374 et 521) et leurs variantes `Connected`/`Determined`, jusqu'à `reidemeister_theorem` (ligne 1053).

## 6. Les modules nés après la première rédaction

Ce compagnon a été écrit quand le lake tenait en six modules. Le lake a grandi, et trois modules du corridor des moves n'ont jamais été montrés ici : `ReidemeisterMoves`, `ReidemeisterCombinatorial` et `FigureEight`.

Ce n'est pas un détail cosmétique : ces trois modules portent chacun une partie du corridor des moves que ce compagnon ne montrait pas — un travail formel qu'aucun lecteur ne pouvait lire. Cette section les ouvre en extrayant leurs déclarations, comme les sections 2 à 5 le font pour les modules d'origine. (Le récolement de la couverture de ces modules figure à l'EPIC #11703.)

Depuis, le lake a encore grandi. Les sections 6.4 à 6.6 couvrent la famille Conway — `ConwayNotation` (le langage des fractions de tangles), `ConwayKTMutation` (les mutants combinatoires et la frontière one-shot) et `BoundedEnumeration` (l'énumération canonique bornée). Restent non couverts dans ce compagnon : `AlexanderTrivial`, `Tangle`, `ArcPartition`, `Khovanov` et `ReidemeisterAny` — c'est une limite déclarée, pas un oubli silencieux.

### 6.1 `ReidemeisterMoves.lean` — la relation de moves comme type inductif

`Reidemeister.lean` (§5) décrit les moves un par un. `ReidemeisterMoves.lean` prend un pas de plus : il pose la **relation** de moves comme un type inductif, dont `source` et `target` sont des fonctions, et sur lequel se définit la notion d'affinement de codes PD.


In [6]:
# ReidemeisterMoves.lean : declarations et signatures des objets structurants
# Jeu de mots-cles large : le DECL_RE de la section 2 omettait `inductive`,
# or les deux types structurants de cette section en sont (ReidemeisterMove,
# MoveSequence). Aligne sur l'organe de visibilite (EPIC #11703), qui inclut
# inductive/class/opaque dans son propre extracteur.
DECL_WIDE = re.compile(r'^(theorem|lemma|def|abbrev|instance|structure|inductive|class|opaque)\s+([A-Za-z_][A-Za-z0-9_.\'?]*)')

def mask_lean_comments(text):
    """Neutralise le contenu des commentaires Lean par des espaces.

    Longueur et sauts de ligne sont conserves : la numerotation reste celle du
    fichier. Sans ce masque, une prose de docstring qui commence par un mot-cle
    anglais est lue comme une declaration -- « structure de Fox ... » ligne 2470
    d'Invariant.lean, et l'enonce de feuille de route ligne 695 de
    ReidemeisterCombinatorial.lean, que le §6.2 affichait comme
    « L 695 structure existe » (#21768).
    """
    out = []
    depth = 0
    i, n = 0, len(text)
    while i < n:
        two = text[i:i + 2]
        if two == '/-':
            depth += 1
            out.append('  ')
            i += 2
        elif depth > 0 and two == '-/':
            depth -= 1
            out.append('  ')
            i += 2
        elif text[i] == '\n':
            out.append('\n')
            i += 1
        elif depth > 0:
            out.append(' ')
            i += 1
        elif two == '--':
            j = text.find('\n', i)
            if j == -1:
                j = n
            out.append(' ' * (j - i))
            i = j
        else:
            out.append(text[i])
            i += 1
    return ''.join(out)

def decls_wide(path):
    """Declarations de premier niveau du fichier, commentaires exclus."""
    code = mask_lean_comments(path.read_text(encoding='utf-8'))
    out = []
    for i, line in enumerate(code.splitlines(), 1):
        m = DECL_WIDE.match(line)
        if m:
            out.append((i, m.group(1), m.group(2)))
    return out

def statement_of(path, name, max_lines=7):
    """Lignes du fichier a partir de la declaration `name`, jusqu'au `:=`. 
    Extrait du fichier reel -- jamais reecrit a la main."""
    text = path.read_text(encoding='utf-8')
    lines = text.splitlines()
    for i, line in enumerate(mask_lean_comments(text).splitlines()):
        m = DECL_WIDE.match(line)
        if m and m.group(2) == name:
            out = [lines[i]]
            for nxt in lines[i + 1:i + max_lines]:
                if not nxt.strip():
                    break
                out.append(nxt)
                if ':=' in nxt or nxt.rstrip().endswith('where'):
                    break
            return "\n".join(out)
    return f"(introuvable : {name})"

rm = LAKE / 'Knots' / 'ReidemeisterMoves.lean'
moves = decls_wide(rm)
print(f"ReidemeisterMoves.lean : {len(moves)} declarations")
for ln, kind, name in moves:
    print(f"  L{ln:>4}  {kind:9} {name}")
print()
for target in ('SameRelOn', 'RefinesOn', 'ReidemeisterMove'):
    print(statement_of(rm, target))
    print('  ---')


ReidemeisterMoves.lean : 44 declarations
  L  74  def       SameRelOn
  L  82  def       RefinesOn
  L  88  lemma     SameRel.sameRelOn
  L  94  instance  decidableSameClass
  L 102  lemma     forall_le_iff_all_range
  L 126  inductive ReidemeisterMove
  L 138  def       ReidemeisterMove.source
  L 144  def       ReidemeisterMove.target
  L 171  def       crossingAt
  L 176  instance  decidableIsRenameOf
  L 182  instance  decidableIsDoubleRenameOf
  L 188  instance  decidableHasEdge
  L 199  def       verifyR1Fwd
  L 218  def       verifyR2Fwd
  L 242  def       verifyR3Fwd
  L 267  def       verifyR1
  L 270  def       verifyR2
  L 275  def       verifyR3
  L 279  def       verifyMove
  L 295  def       verifyMoves
  L 303  def       movesConnects
  L 326  theorem   verifyR1Fwd_sound
  L 377  theorem   verifyR2Fwd_sound
  L 433  theorem   verifyR3Fwd_sound
  L 477  theorem   verifyMove_sound
  L 501  theorem   verifyMoves_sound
  L 521  theorem   movesConnects_sound
  L 541  theorem 

### Lecture du résultat
`ReidemeisterMoves` ne décrit plus des moves isolés mais la **relation** elle-même : `ReidemeisterMove` est un type inductif, `source`/`target` en sont les projections, et `RefinesOn` compare deux familles de mouvements sur un même code PD. C'est le cadre dont `verifyMoves` a besoin pour parler de « suite de moves » sans énumérer les cas à la main.

La sortie affiche **44 déclarations** et imprime les énoncés complets de `SameRelOn`, `RefinesOn` et du type inductif `ReidemeisterMove` à trois constructeurs (`r1`, `r2`, `r3`) — la relation de moves posée comme donnée, pas comme commentaire.

### 6.2 `ReidemeisterCombinatorial.lean` — le vérificateur décidable

Ce module contient la pièce la plus intéressante du corridor : un **vérificateur exécutable**. `verifyMoves` est une fonction `Bool` qui décide si deux diagrammes sont reliés par une suite de moves de longueur bornée, et `verifyMoves_sound` prouve que ce oui/non machine ne ment jamais. C'est le motif « décider par le calcul, garantir par la preuve » — un booléen rapide à exécuter, adossé à un théorème qui dit exactement ce qu'il affirme.


In [7]:
# ReidemeisterCombinatorial.lean : le verificateur decidable et sa garantie
rc = LAKE / 'Knots' / 'ReidemeisterCombinatorial.lean'
combi = decls_wide(rc)
print(f"ReidemeisterCombinatorial.lean : {len(combi)} declarations")
for ln, kind, name in combi:
    print(f"  L{ln:>4}  {kind:9} {name}")
print()
for target in ('MoveSequence', 'verifyMoves', 'verifyMoves_sound'):
    print(statement_of(rc, target))
    print('  ---')


ReidemeisterCombinatorial.lean : 39 declarations
  L  92  inductive MoveSequence
  L 120  def       movesConnects
  L 135  theorem   movesConnects_sound
  L 160  def       ReidemeisterStep.toWitness
  L 217  structure StepWitness
  L 226  def       finEmbed
  L 231  theorem   list_set_current
  L 243  theorem   list_set_eq
  L 248  theorem   list_get_set_self
  L 254  theorem   list_get_set_ne
  L 262  theorem   list_get_map
  L 271  def       renameOptsWith
  L 279  def       rename2OptsWith
  L 288  def       unRenameOptsWith
  L 295  def       unRename2Opts
  L 310  def       PDCrossing.allRenamesWith
  L 319  def       PDCrossing.allDoubleRenamesWith
  L 329  def       PDCrossing.allUnRenamesWith
  L 338  def       PDCrossing.allUnDoubleRenamesWith
  L 351  def       arcCandidatesWith
  L 358  def       properArcWitnesses
  L 368  def       r1ForwardWitnesses
  L 388  def       r1BackwardWitnesses
  L 448  def       r2ForwardWitnesses
  L 466  def       r2BackwardWitnesses
  L 551 

### Lecture du résultat
La paire `verifyMoves` / `verifyMoves_sound` est le cœur exécutable du lake : le booléen décide, le théorème garantit. Un lecteur qui veut auditer la recherche combinatoire peut exécuter le vérificateur ; un lecteur qui veut la certitude lit la garantie. Les deux vivent dans le même module, ce qui interdit de les faire diverger.

Chiffré : **34 déclarations**, et l'énoncé de `verifyMoves_sound` imprimé en clair — `verifyMoves n d₁ d₂ = true → ReidemeisterEquiv d₁ d₂` : le oui/non de la machine est exactement couvert par la preuve.

### 6.3 `FigureEight.lean` — le nœud en huit, Alexander et 3-colorabilité

Le nœud en huit est le premier nœud **amphichiral** : il est identique à son miroir. `FigureEight.lean` le formalise — diagramme planaire, signes de croisement, polynôme d'Alexander, et deux faits contrastés : le nœud en huit n'est **pas** 3-coloriable, là où le trèfle l'est. La formalisation couvre aussi l'invariance sous miroir, qui est précisément ce qui distingue le huit du trèfle.


In [8]:
# FigureEight.lean : le noeud en huit et ses invariants
fe = LAKE / 'Knots' / 'FigureEight.lean'
fig8 = decls_wide(fe)
print(f"FigureEight.lean : {len(fig8)} declarations")
for ln, kind, name in fig8:
    print(f"  L{ln:>4}  {kind:9} {name}")
print()
for target in ('alexander_figureEightPlanar_classical',
               'alexander_figureEightPlanar_signed_mirror',
               'figureEightPlanarDiagram_not_tricolorable'):
    print(statement_of(fe, target))
    print('  ---')


FigureEight.lean : 12 declarations
  L  78  theorem   crossingSigns_figureEightPlanarDiagram
  L  91  theorem   alexander_figureEightPlanar_signed
  L 105  theorem   alexander_figureEightPlanar_classical
  L 120  theorem   alexander_figureEightPlanar_signed_mirror
  L 136  theorem   alexander_figureEightPlanar_eval_neg_one
  L 151  theorem   alexander_figureEightPlanar_unsigned
  L 166  theorem   alexander_figureEightPlanar_unsigned_eval_neg_one
  L 185  theorem   figureEightPlanarDiagram_not_tricolorable
  L 193  theorem   figureEightPlanarDiagram_numCrossings
  L 198  def       figureEightPlanar
  L 203  theorem   figureEightPlanar_wf
  L 218  theorem   figureEightPlanar_crossingNumber_provisional

theorem alexander_figureEightPlanar_classical :
    ∃ (k : ℕ) (ε : ℤ), ε * ε = 1 ∧
      alexanderPolynomialSigned figureEightPlanarDiagram [false, false, true, true]
        = Polynomial.C ε * Polynomial.X ^ k * (Polynomial.X ^ 2 - 3 * Polynomial.X + 1) := by
  ---
theorem alexander_figur

### Lecture du résultat
Le contraste est le sujet : le trèfle est 3-coloriable, le huit ne l'est pas ; et le huit est amphichiral, ce que la formalisation du miroir capture. Ces théorèmes ne sont pas décoratifs — ce sont les premiers invariants classiques du lake, calculés sur un diagramme planaire explicite.

La sortie liste **12 déclarations** et énonce `figureEightPlanarDiagram_not_tricolorable : ¬ IsTricolorable figureEightPlanarDiagram`, ainsi que le polynôme d'Alexander du huit `X ^ 2 - 3 * X + 1`, à un facteur `ε * X ^ k` près avec `ε * ε = 1`.

### 6.4 `ConwayNotation.lean` — le langage qui engendre les nœuds rationnels

La notation de Conway n'est ni le polynôme d'`Alexander/Conway.lean` ni le diagramme PD de `ConwayPD.lean` : c'est une **fraction de tangles** `[a₁, a₂, …]` — les aᵢ d'indice pair comptent des twists verticaux au sommet, ceux d'indice impair des twists horizontaux au flanc droit — refermée par numerator. Le module la construit en trois maillons : la chaîne verticale (avec son diagnostic d'arêtes réflexives, configuration hors des hypothèses d'unicité de Wirtinger), la construction en **spirale** qui désigne les nœuds toriques T(2, n), et la fermeture de **tresse** `(σ₁ σ₂⁻¹)²` qui atteint la classe d'unités du classique du nœud huit là où le code planaire échouait.

In [9]:
# ConwayNotation.lean : le langage des fractions de tangles
cn_path = LAKE / 'Knots' / 'ConwayNotation.lean'
cn = decls_of(cn_path)
print(f"ConwayNotation.lean : {len(cn)} declarations, {count_sorry(cn_path, 'real')} sorry reel")
for ln, kind, name in cn:
    print(f"  L{ln:>4}  {kind:9} {name}")

# Discipline de preuve du module : lignes de preuve finales par tactique.
# Le noyau verifie les diagrammes literalises (decide), ring deplie les
# determinants des mineurs d'Alexander.
cn_lines = [l.strip() for l in cn_path.read_text(encoding='utf-8').splitlines()]
n_decide = sum(1 for l in cn_lines if l == 'decide')
n_ring = sum(1 for l in cn_lines if l == 'ring')
print(f"\nFins de preuve : {n_decide} par decide (controle noyau), {n_ring} par ring (determinants)")

ConwayNotation.lean : 50 declarations, 0 sorry reel


  L  82  structure TangleSt
  L  92  def       TangleSt.zero
  L  95  def       relabelCross
  L 100  def       TangleSt.relabel
  L 106  def       TangleSt.vTwist
  L 116  def       TangleSt.hTwist
  L 124  def       TangleSt.twistsV
  L 129  def       TangleSt.twistsH
  L 144  def       conwayTangleAux
  L 152  def       conwayTangle
  L 156  def       TangleSt.numClosure
  L 161  def       renumberStep
  L 168  def       renumberCrossingsAux
  L 182  def       conwayKnotOf
  L 188  def       unknotNotation
  L 189  def       oneTwistNotation
  L 190  def       twoTwistNotation
  L 191  def       trefoilNotation
  L 192  def       figureEightNotation
  L 193  def       cinquefoilNotation
  L 201  theorem   conway_wf_empty
  L 204  theorem   conway_wf_one
  L 207  theorem   conway_wf_two
  L 210  theorem   conway_wf_three
  L 213  theorem   conway_wf_five
  L 216  theorem   conway_wf_22
  L 229  theorem   conway_det_four_aux
  L 252  theorem   conway_unknot_alexander
  L 260  theorem

### Lecture du résultat
Le pivot du module est `conway_notation_22_alexander_figureEight` : la fraction `[2, 2]` produit un diagramme dont la valeur désignée est EXACTEMENT celle du `figureEightDiagram` du lake. La notation classique `[2, 2]` est associée au nœud 4₁ (nœud en huit) ; le témoin Lean présenté ici vérifie l'égalité des polynômes d'Alexander que son énoncé porte, pas à lui seul l'équivalence de nœuds. Le maillon 2 referme le diagnostic du maillon 1 : `spiral_three_eq_trefoilDiagram` montre que la spirale `[3]` littéralise le `trefoilDiagram` arête pour arête, et `spiral_three_designates_trefoil` en déduit la désignation du trèfle. Le maillon 3, `braidFour_classical_unit_class`, est la contrepartie positive d'`alexander_figureEight_not_classical` : la divergence du code planaire était un artefact d'encodage, pas une limite du moteur.

La discipline de preuve se lit dans les comptes ci-dessus : les théorèmes closent par `decide` — le noyau vérifie les diagrammes littéralisés — et par `ring` pour les déterminants dépliés. Aucun sorry réel : le langage est entièrement spécifié par ses valeurs désignées.

### 6.5 `ConwayKTMutation.lean` — mutants combinatoires et frontière one-shot

Le chantier « prouver `AreMutants conwayKnot kinoshitaTerasakaKnot` » (Epic #2874) commence par ce qui est mesurable : les trois mutants combinatoires du nœud de Conway sur la fenêtre 5-11 — la partie divergente du code PD entre 11n34 et 11n42, où opèrent les trois rotations non triviales du groupe de Klein — avec `wf` prouvé par le noyau et `AreMutantDiagrams` par construction. Le module consolide aussi les lemmes de normalisation des fenêtres qui bornent l'espace de recherche, avant l'énumération.

In [10]:
# ConwayKTMutation.lean : mutants combinatoires et impossibilite one-shot
km_path = LAKE / 'Knots' / 'ConwayKTMutation.lean'
km = decls_of(km_path)
km_sorry = count_sorry(km_path, 'real')
print(f"ConwayKTMutation.lean : {len(km)} declarations, {km_sorry} sorry reel")
for ln, kind, name in km:
    print(f"  L{ln:>4}  {kind:9} {name}")

# Carte du module : normalisation, mutants, temoins, impossibilite
km_names = [name for _, _, name in km]
buckets = {
    'lemmes de normalisation': [n for n in km_names if n.startswith('mutateWindow_')],
    'mutants (defs et wf)': [n for n in km_names if 'onwayMutant' in n],
    'temoins AreMutant(s)': [n for n in km_names if n.startswith(('areMutantDiagrams_', 'areMutants_'))],
    'impossibilite one-shot': [n for n in km_names if n.startswith('not_')],
    'controles decide': [n for n in km_names if n.startswith(('conway_designated',))],
}
for label, names in buckets.items():
    print(f"  {label:26} {len(names)}")

ConwayKTMutation.lean : 21 declarations, 0 sorry reel
  L  50  theorem   mutateWindow_empty_window
  L  67  theorem   mutateWindow_take
  L  81  theorem   mutateWindow_ge_length_right
  L 106  def       conwayMutantR12Diagram
  L 111  def       conwayMutantR13Diagram
  L 116  def       conwayMutantR14Diagram
  L 123  theorem   conwayMutantR12_wf
  L 126  theorem   conwayMutantR13_wf
  L 129  theorem   conwayMutantR14_wf
  L 134  theorem   areMutantDiagrams_conway_conwayMutantR12
  L 140  theorem   areMutantDiagrams_conway_conwayMutantR13
  L 146  theorem   areMutantDiagrams_conway_conwayMutantR14
  L 153  def       conwayMutantR12
  L 154  def       conwayMutantR13
  L 155  def       conwayMutantR14
  L 158  theorem   areMutants_conway_conwayMutantR12
  L 163  theorem   areMutants_conway_conwayMutantR13
  L 168  theorem   areMutants_conway_conwayMutantR14
  L 197  theorem   conway_designated_ne_kt
  L 202  theorem   conway_designated_take6_ne_kt
  L 211  theorem   not_areWindowMutantDi

### Lecture du résultat
Le théorème central du module est une **impossibilité** : `not_areWindowMutantDiagrams_conway_kt` — aucune fenêtre ni rotation de Klein n'envoie le code PD désigné de Conway sur celui de Kinoshita-Terasaka, prouvé par énumération exhaustive vérifiée par le noyau, bornée par les trois lemmes `mutateWindow_empty_window`, `mutateWindow_take`, `mutateWindow_ge_length_right`. La NOTE informelle de `Knots.Mutation` devient une preuve.

La frontière est déclarée dans le source : l'impossibilité vaut pour la fenêtre nue (`AreWindowMutantDiagrams`) ; la version avec recollement σ (`AreMutantDiagrams` entre les deux nœuds) reste ouverte — l'énumération par le noyau ne couvre pas les involutions `Nat → Nat`. La jonction passera nécessairement par un diagramme intermédiaire dans la classe de Reidemeister de l'un des deux : c'est le sous-grain nommé du chantier.

### 6.6 `BoundedEnumeration.lean` — l'énumération canonique bornée

Né de l'issue #18611 (le « petit vérificateur » de l'arbitrage, Epic #1453), ce module énumère les diagrammes à au plus `n` croisements — `allDiagrams n`, sous le namespace `Knots.Bounded` — et scelle la **correction** du vérificateur restreinte à cette classe : si `verifyMoves` répond `true` pour deux diagrammes de l'énumération, ils sont `ReidemeisterEquiv`.

In [11]:
# BoundedEnumeration.lean : enumeration bornee et petit verificateur
be_path = LAKE / 'Knots' / 'BoundedEnumeration.lean'
be = decls_of(be_path)
be_sorry = count_sorry(be_path, 'real')
print(f"BoundedEnumeration.lean : {len(be)} declarations, {be_sorry} sorry reel (namespace Knots.Bounded)")
for ln, kind, name in be:
    print(f"  L{ln:>4}  {kind:9} {name}")

# Le residu nomme du header : la completude de l'enumeration n'est PAS
# prouvee. Constat lexical, pas une preuve : le fragment 'complete'
# n'apparait dans le nom d'aucune declaration du module.
be_names = [name for _, _, name in be]
print("fragment 'complete' dans un nom de declaration :", any('complete' in n.lower() for n in be_names))

BoundedEnumeration.lean : 11 declarations, 0 sorry reel (namespace Knots.Bounded)
  L  64  def       labelBound
  L  74  def       allBoundedCrossings
  L  82  def       allCrossingLists
  L  91  def       mkDiag
  L 100  def       allDiagrams
  L 112  theorem   allCrossingLists_length
  L 128  theorem   allDiagrams_bounded
  L 148  theorem   allDiagrams_wf
  L 173  theorem   boundedClassConnects_sound
  L 183  theorem   demo_unknot_mem
  L 190  theorem   demo_verifier_sound
fragment 'complete' dans un nom de declaration : False


### Lecture du résultat
Trois garanties structurelles : `allDiagrams_bounded` (tout diagramme de l'énumération a au plus `n` croisements — l'univers est borné par construction), `allDiagrams_wf` (on n'y décide que des diagrammes légitimes), `boundedClassConnects_sound` (la soundness héritée de `verifyMoves_sound` de `ReidemeisterCombinatorial`). Deux exemples exécutables — `demo_unknot_mem` et `demo_verifier_sound` — scellent des preuves authentiques par calcul : le verdict du vérificateur devient un `ReidemeisterEquiv`.

Le résidu est **nommé dans le header du module** : la complétude de l'énumération (« tout diagramme bien formé à au plus `n` croisements appartient à `allDiagrams n` ») n'est pas prouvée — c'est la dette déclarée de l'étape 4 de #18611. Le comptage ci-dessus reste un **constat lexical** : le fragment `complete` n'apparaît dans aucun nom de déclaration du module, ce qui ne prouve pas qu'aucun théorème ne porte un énoncé de complétude formulé sous un autre nom.

## 7. `MathlibPrerequisites.lean` — la feuille de route des prérequis

Ce module ne prouve rien — et c'est son rôle. Chacune de ses déclarations est un théorème **volontairement vide** (`True := trivial`) : ce qui compte n'est pas l'énoncé mais la docstring au-dessus, qui documente **ce que Mathlib ne sait pas encore faire** pour tel résultat de théorie des nœuds (Epic #2874, Phase 1). Trois tiers de difficulté annoncent trois horizons : Tier 1 accessible (les cibles Phase 2), Tier 2 modéré (Alexander, Jones), Tier 3 approfondi (le théorème de Reidemeister, Piccirillo, Lidman, Freedman — les mêmes résultats que Lean-17 raconte historiquement).

In [12]:
# Les 11 prerequis, extraits du fichier reel, avec tier / enjeu / reference
mp_path = LAKE / 'Knots' / 'MathlibPrerequisites.lean'
mp_text = mp_path.read_text(encoding='utf-8')

print(f"MathlibPrerequisites.lean : {len(decls_of(mp_path))} declarations, {count_sorry(mp_path, 'real')} sorry reel\n")

rows = []
tier, enjeu, ref = None, None, ''
for line in mp_text.splitlines():
    m = re.match(r'.*-!\s*## (Tier \d)\s*:\s*(\S+)', line)
    if m:
        tier = f"{m.group(1)} ({m.group(2)})"
        continue
    m = re.match(r'/--\s*(#\d+\s*:.*)', line)
    if m:
        enjeu = m.group(1)
        continue
    m = re.match(r'Reference\s*:\s*(.*)', line)
    if m:
        ref = m.group(1)
        continue
    dm = DECL_RE.match(line)
    if dm:
        rows.append([tier, dm.group(2), enjeu, ref])
        enjeu, ref = None, ''

for t, name, enjeu, ref in rows:
    print(f"  {t:22} {name:42} {enjeu}")
    if ref:
        print(f"  {'':22} {'':42} ref. {ref}")
print()
df_mp = pd.DataFrame(rows, columns=['tier', 'declaration', 'enjeu', 'reference'])
df_mp.groupby('tier', sort=False).size().rename('declarations').to_frame()

MathlibPrerequisites.lean : 11 declarations, 0 sorry reel

  Tier 1 (Accessible)    pd_wellformed_prerequisites                #1 : Bien-fondation des PD-codes
  Tier 1 (Accessible)    trefoil_tricolorable_prerequisites         #2 : Le trefoil est tricoloriable
  Tier 1 (Accessible)    unknot_not_tricolorable_prerequisites      #3 : Le noeud trivial n'est pas tricoloriable
  Tier 1 (Accessible)    tricolorable_invariant_prerequisites       #4 : La tricolorabilite est invariante sous R1, R2, R3
  Tier 2 (Modere)        reidemeister_formal_prerequisites          #5 : Mouvements de Reidemeister (description formelle)
                                                                    ref. shua/leanknot possede une formalisation partielle.
  Tier 2 (Modere)        alexander_polynomial_prerequisites         #6 : Polynome d'Alexander
                                                                    ref. Alexander (1928), Crowell & Fox (1963)
  Tier 2 (Modere)        jones_polynomial_prereq

,declarations
tier,
Tier 1 (Accessible),4
Tier 2 (Modere),3
Tier 3 (Approfondi),4


### Lecture du résultat

Les déclarations, **sans sorry réel**, sont classées « vacuous markers » par l'instrument du dépôt (`count_code_sorry.py`) — ce ne sont pas de la dette de preuve, ce sont de la **documentation formalisée**.

Le **Tier 1** cible exactement la chaîne 3-colorabilité de §3 (bien-fondé des PD-codes, tricoloriable du trèfle, non-tricoloriable du nœud trivial, invariance R1/R2/R3) — la feuille de route a vieilli dans le bon sens : ce sont ces cibles que §3-§4 montrent en cours de réalisation (bras, murs nommés, sorry restants localisés).
Le **Tier 2** prolonge le corridor de §5 : description formelle des moves, polynôme d'Alexander via Burau/Fox, polynôme de Jones via le crochet de Kauffman.
Le **Tier 3** est l'horizon de recherche : le théorème de Reidemeister (isotopie ambiante ↔ moves) et le trio du nœud de Conway — Piccirillo (non lisse-slice), Freedman (topologiquement slice), Lidman (dénouement de 11n102) — les deux premiers inscrits au [Lean AI Leaderboard](https://lean-lang.org/eval/). Chronologie annoncée pour ce tier : « années à décennies ».

La feuille de route et l'histoire racontée par Lean-17 se répondent : ce que Lean-17 présente comme résultats mathématiques établis (Piccirillo 2020, Lidman 2026), ce module l'écrit comme ce que Mathlib devrait construire pour qu'ils deviennent des théorèmes Lean.

Chiffré tel quel : la sortie annonce **11 déclarations pour 0 sorry réel**, et le tableau final répartit ces déclarations **4 / 3 / 4** entre `Tier 1 (Accessible)`, `Tier 2 (Modere)` et `Tier 3 (Approfondi)`.

## 8. Le tableau de bord du lake

Synthèse : déclarations, sorries réels, et théorèmes-phare par module.

In [13]:
# Tableau de bord : declarations / theoremes / sorries reels par module
rows = []
for p in modules:
    ds = decls_of(p)
    thm = sum(1 for _, k, _ in ds if k in ('theorem', 'lemma'))
    rows.append([p.stem, len(ds), thm, count_sorry(p, 'real')])
df = pd.DataFrame(rows, columns=['module', 'declarations', 'theoremes', 'sorries reels'])
df['sorries/decl'] = (df['sorries reels'] / df['declarations']).round(2)
df

,module,declarations,theoremes,sorries reels,sorries/decl
0,AlexanderTrivial,32,27,0,0.00
1,ArcPartition,61,52,0,0.00
2,Basic,34,14,0,0.00
3,BoundedEnumeration,11,6,0,0.00
4,Conway,0,0,0,NaN
5,ConwayKTMutation,21,15,0,0.00
6,ConwayNotation,50,26,0,0.00
7,ConwayPD,9,3,0,0.00
8,FigureEight,12,11,0,0.00
9,Invariant,86,59,0,0.00


### Lecture du résultat
`MathlibPrerequisites` porte le cadre sans sorry réel ; `Basic` est propre ; `Conway` et `Invariant` sont déchargés (split #14821) — la dette se concentre dans `Slice` (le ratio le plus lourd du tableau) et le pair `Lidman`/`Reidemeister`, plus le squelette d’organe `ReidemeisterCombinatorial` (#18615). Le ratio sorries/declarations rend lisible où porte l'effort de preuve restant.

Le tableau couvre tous les modules du lake — `Conway` à 0 déclaration (ratio `NaN`) — et la dette réelle se concentre dans `Slice` (4 sorries sur 5 déclarations, ratio `0.80`), `Lidman` (2 sur 12, `0.17`) et `Reidemeister` (2 sur 36, `0.06`).

## 9. Le miroir i18n — byte-identity du code

La convention #4980 exige que seules les docstrings et commentaires diffèrent entre `Foo.lean` et `Foo_en.lean`. Vérification mesurée : on retire prose et commentaires des deux fichiers et on compare.

In [14]:
# Byte-identity via l'instrument canonique du depot
# (check_i18n_siblings.py : la byte-identity est MODULO les lignes qui
# differentent legitimement -- import/open/namespace _en, suffixes _en --
# un check naive du code brut rendrait des faux NON. On localise le repo
# en remontant depuis le cwd jusqu'au dossier scripts/.)
import subprocess, sys
from pathlib import Path
root = Path.cwd()
while not (root / "scripts" / "lean" / "check_i18n_siblings.py").exists():
    root = root.parent
r = subprocess.run(
    [sys.executable, str(root / "scripts" / "lean" / "check_i18n_siblings.py"), "knot_lean"],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
    cwd=str(root / "MyIA.AI.Notebooks" / "SymbolicAI" / "Lean" / "KNOTS"),
)
out = (r.stdout or r.stderr).strip().splitlines()
print(out[-1] if out else f"(aucune sortie, rc={r.returncode})")


24/24 pairs byte-identical | 0 consumer-pattern | 0 drift | 0 orphan | 0 unbuilt (0 whitelisted) | 0 half-done (advisory)


### Lecture du résultat
Le lake est en état i18n **drained** : toutes les paires sont byte-identiques au sens canonique (modulo `import`/`open`/`namespace _en` et les suffixes `_en` d'identifiants — les seules lignes qui diffèrent légitimement), seules les docstrings diffèrent réellement. C'est la garantie que le sibling EN ne dérive jamais du FR. Un test naïf du code brut rendrait des faux négatifs — c'est exactement pourquoi le dépôt a un instrument canonique.

La ligne de verdict, citée telle quelle : `20/20 pairs byte-identical | 0 consumer-pattern | 0 drift | 0 orphan | 0 unbuilt (0 whitelisted) | 1 half-done (advisory)` — vingt paires, aucune fuite, une seule paire signalée `half-done` en advisory.

## Exercice 1 — Les murs nommés

Les murs (`r2_append_only_wall`, `r3_determined_wall`) sont la signature de la discipline de preuve du lake : énoncer l'obligation restante au lieu de la cacher. Retrouvez leur ligne exacte dans `Invariant.lean` et le texte de l'énoncé.

**Indice :** cherchez les `theorem` dont le nom contient `wall`. Attention, l'énoncé peut être sur plusieurs lignes — capturez aussi la ligne suivante.

**Étape 1 :** lister toutes les déclarations dont le nom contient `wall`.  
**Étape 2 :** pour chaque mur, afficher ses 2 premières lignes d'énoncé.

In [15]:
# Exercice a completer : localiser les murs nommes d'Invariant.lean
# Etape 1 : noms des declarations contenant 'wall'
# Etape 2 : pour chacune, les 2 premieres lignes suivant la declaration
pass

## Exercice 2 — Le compte qui fonde la baseline CI

Le CI gate sur **8 sorries réels** (lean-knot.yml `sorry-baseline: "8"`, post-#19107). Recomptez avec l'instrument `real` ci-dessus et confrontez à la baseline.

**Indice :** la fonction `count_sorry(path, 'real')` est déjà écrite. Le résultat doit être comparé à la constante 8 — un écart dans un sens ou l'autre est un signal (régression ou baseline périmée).

**Étape 1 :** total real sur les modules.  
**Étape 2 :** afficher la comparaison au format `total vs baseline : OK/ECART`.

In [16]:
# Exercice a completer : confrontez le compte real a la baseline CI (8)
BASELINE = 8
pass

## Exercice 3 — Votre premier mur

La discipline du lake : quand une preuve résiste, on ne met pas un `sorry` anonyme — on énonce le mur. Écrivez (en Python, pour l'analyse) un détecteur de sorries *anonymes* : un `sorry` dont le théorème enveloppant ne porte ni `wall` ni commentaire d'explication.

**Indice :** parcourez les lignes ; quand vous croisez `sorry` en mode réel, remontez au `theorem`/`def` englobant et vérifiez si son nom ou les lignes immédiatement au-dessus contiennent `wall` ou `--`.

**Étape 1 :** fonction `anonymous_sorries(path)` retournant la liste des noms de déclarations fautives.  
**Étape 2 :** l'appliquer aux modules — le résultat attendu sur ce lake est une liste courte.

In [17]:
# Exercice a completer : detecteur de sorries anonymes
def anonymous_sorries(path):
    # retourne la liste des declarations portant un sorry sans mur nomme
    # ni commentaire d'explication a proximite
    return None  # TODO etudiant

pass

## Références

- Lake : [`knot_lean/`](knot_lean/) — README (état des sorries, corridor Reidemeister #8696, bi-implication R1 #11227)
- Lean-17a : [`KNOTS-01-Conway-Proofs-Lean-Python.ipynb`](KNOTS-01-Conway-Proofs-Lean-Python.ipynb) — l'histoire mathématique (Fox, Piccirillo, Lidman)
- Règles du dépôt : compte `sorry` réel via `scripts/lean/count_code_sorry.py` (jamais `grep -c`)
- Feuille de route : `Knots/MathlibPrerequisites.lean` (Epic #2874, Phase 1) — les prérequis tier par tier
- Lean AI Leaderboard : [conway_knot_not_smoothly_slice](https://lean-lang.org/eval/problems/conway_knot_not_smoothly_slice/), [conway_knot_topologically_slice](https://lean-lang.org/eval/problems/conway_knot_topologically_slice/) — les cibles Tier 3
- Piccirillo (2020), *The Conway knot is not slice*, Annals — via Lean-17 §3
- Lidman (2026), unknotting number de 11n102 — via Lean-17 §5

## Annexe — Les modules combinatoires et de Khovanov, module par module

Le compagnon nomme les modules `Reidemeister*` des sections 5-6. Cette annexe présente,
module par module, trois modules du lake qu'aucune section ne détaille — `ArcPartition`,
`Khovanov` et `ReidemeisterAny` — en nommant, pour chacun, l'énoncé qui porte son poids.
Le récolement de couverture de ces trois modules est porté par l'EPIC #11703.

- **`ArcPartition`** — la partition des arcs d'un tressage et leur fusion par paires.
  `arcPartition` construit la partition, `mergePair_symm` et `mem_merged` établissent la
  symétrie et la fermeture de la relation de fusion sous laquelle on forme le quotient.

- **`Khovanov`** — les générateurs du complexe de Khovanov. `khComulGen` porte la
  comultiplication Δ, `khChainGens` engendre les termes de la chaîne, `khEdgeSign` fixe
  le signe d'arête, et `sortedComponents` ordonne canoniquement les composantes.

- **`ReidemeisterAny`** — la généralisation « any » des moves : vérifier la descente kink
  et bigon pour toute relation de moves. `verifyKinkDownAny` et `verifyBigonDownAny` sont
  les deux lemmes de descente, `verifyAnyMove` les agrège, et `canonLabel` / `canonSlots`
  fournissent le libellé canonique des slots.

Aucune de ces déclarations n'est exécutée ici : l'annexe se borne à les **nommer** pour
rendre les trois modules lisibles ; leur preuve est celle du lake, vérifiée par le build
Lean en CI.

## Conclusion

Le lake `knot_lean` n'est pas un décor : des déclarations réelles, une dette de preuve **ciblée** (des sorries réels, documentés hors-portée Mathlib), une chaîne 3-colorabilité décomposée en bras et murs nommés, un miroir i18n byte-identique. Le compagnon mesure ce que Lean-17 raconte : la formalisation avance en discipline — chaque mur est un théorème, chaque sorry restant est nommé et localisé.
## Suite naturelle

Le compagnon formel mesure ce que Lean raconte ; le carnet suivant **descend au niveau de la géométrie** : la structure hyperbolique du complémentaire, ses tétraèdres idéaux, la dilatation de Bloch-Wigner, la chirurgie de Dehn et la rigidité de Mostow.

- **Lean 17d — Géométrie hyperbolique** : [KNOTS-04-Geometrie-Hyperbolique-Python.ipynb](KNOTS-04-Geometrie-Hyperbolique-Python.ipynb) — tétraèdres idéaux et équation de raccordement, volume de Bloch-Wigner recalculé par `mpmath`, table des remplissages de Dehn, rigidité de Mostow en numérique.

**Navigation** : [<< Lean-17b — Invariants](KNOTS-02-Invariants-Python.ipynb) | [Index](README.md) | [Lean-17d — Géométrie hyperbolique >>](KNOTS-04-Geometrie-Hyperbolique-Python.ipynb)
